<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/12_MCP_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**What this notebook demonstrates**

This notebook shows a minimal end-to-end setup of the Model Context Protocol (MCP) using a simple tool, all within a single Google Colab environment.

It also shows how to call an LLM (OpenAI) with tool support.

**MCP Server (tool provider)**
We define a small MCP server (mcp_server.py) using FastMCP.
The server exposes functions like:

*   add_numbers(a, b)
*   greet(name)

This server communicates over stdio (stdin/stdout) using JSON-RPC.

**Running the server inside Colab**
Since MCP requires a separate process, we:
*   Write the server code to a file using %%writefile
*   Launch it via subprocess.Popen(...)

This effectively simulates how MCP servers run in real systems (as standalone processes).

**MCP Client (inside another cell)**

Another cell acts as a client
It sends JSON-RPC messages to the server via:
*   stdin → request
*   stdout → response

We manually implement a small helper (send_rpc) to communicate with the server

**MCP workflow demonstrated**

The notebook walks through the standard MCP lifecycle:

*   Initialize session
*   List available tools
*   Call a tool
*   Parse the structured response

**Key idea**

Even though everything runs in one notebook, we are still maintaining the core MCP architecture:

*   Server → separate process
*   Client → communicates via stdio
*   Protocol → JSON-RPC

This mirrors real-world usage (e.g., IDEs, LLM agents calling tools), just compressed into a single Colab environment for experimentation.

In [ ]:
#Step 1 — Install MCP SDK
!pip install -q mcp

In [ ]:
#Step 2 — Create a simple MCP server

%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")

# Define a simple tool
@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    return a + b

@mcp.tool()
def greet(name: str) -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

In [ ]:
#Step 3 — Run server as subprocess (important)

import subprocess
import json
import time

# Start MCP server
proc = subprocess.Popen(
    ["python", "mcp_server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1
)

In [ ]:
#Step 4 — Initialize MCP session
#   MCP uses JSON-RPC 2.0 over stdio, so communication looks like:
#   {"jsonrpc": "2.0", "id": 1, "method": "...", "params": {...}}

def send_rpc(method, params=None, id=1):
    msg = {
        "jsonrpc": "2.0",
        "id": id,
        "method": method,
        "params": params or {}
    }
    proc.stdin.write(json.dumps(msg) + "\n")
    proc.stdin.flush()

    # Read response
    return json.loads(proc.stdout.readline())

In [ ]:
#Step 5 — Handshake

response = send_rpc("initialize", {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "colab-client", "version": "1.0"}
})

print(response)

In [ ]:
#Step 6 — List tools

tools = send_rpc("tools/list")
print(tools)


In [ ]:
#Step 7 — Call tool

response = send_rpc("tools/call", {
    "name": "add_numbers",
    "arguments": {"a": 100, "b": 9999}
})

print(response)

In [ ]:
value = response["result"]["structuredContent"]["result"]
print(value)

In [ ]:
#Try another tool

response = send_rpc("tools/call", {
    "name": "greet",
    "arguments": {"name": "Sachin"}
})
print(response)

In [ ]:
value = response["result"]["content"][0]["text"]
print(value)

**Cleaner approach**

Instead of manual JSON-RPC we should use MCP client

**What changes?**

Keep:

*   ✅ Same mcp_server.py
*   ✅ Same subprocess idea (server runs separately)

**Replace:**

*   ❌ Manual send_rpc
*   ❌ JSON parsing

**With:**

*   ✅ High-level client methods (list_tools, call_tool)

**Problem**

This does not work smoothly in Google Colab .The stdio_client relies on real OS-level stdin/stdout/stderr file descriptors, but Google Colab (Jupyter) uses wrapped streams that don’t support methods like fileno(), causing subprocess creation to fail. As a result, async stdio-based MCP clients break in Colab, while manual **subprocess.Popen** works because it explicitly creates proper pipe-backed file descriptors.

**Let's connect our working MCP server to OpenAI so the model calls add_numbers automatically**

User question

   ↓

OpenAI (decides tool call)

   ↓

Your MCP client (send_rpc)

   ↓

MCP server (add_numbers)

   ↓

Result → OpenAI → final answer


In [ ]:
#Step 1 — Define tool schema for OpenAI
tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two integers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    }
]

In [ ]:
!pip install -q openai

In [ ]:
#Setup an LLM

from openai import OpenAI
from google.colab import userdata

#define constants
MODEL = "gpt-4.1-mini"
#MODEL = "gpt-5.2"

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

In [ ]:
#Step 2 — OpenAI call (with tool support)

messages = [
    {"role": "user", "content": "What is 555 + 7777?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

In [ ]:
#Step 3 — Detect tool call

msg = response.choices[0].message
#print(msg)

if msg.tool_calls:
    tool_call = msg.tool_calls[0]

    tool_name = tool_call.function.name
    arguments = eval(tool_call.function.arguments)

    print(f"tool name is :{tool_name}")
    print(f"arguments are :{arguments}")

In [ ]:
#Step 4 — Call my MCP tool

mcp_resp = send_rpc("tools/call", {
    "name": tool_name,
    "arguments": arguments
})

result = mcp_resp["result"]["structuredContent"]["result"]
print(f"result is : {result}")

In [ ]:
#Step 5 — Send result back to model

messages.append(msg)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})

print(messages)

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)



In [ ]:
print(final_response.choices[0].message.content)